# Lab: Minimax Search with Alpha-Beta Pruning

## Case Study: Network Attack vs Defense Simulation

### Scenario

You are designing an AI system that simulates a **cybersecurity battle** inside a computer network.

Two automated agents interact with the system.

Player A (MAX player) → **Attacker**

Player B (MIN player) → **Defender**

The attacker tries to **gain control of the network**, while the defender tries to **protect it**.

Each turn, a player performs an action that changes the state of the network.

Examples of actions include:

- scanning the network
- exploiting vulnerabilities
- patching systems
- blocking access

The game runs for a **fixed number of rounds**.

At the end of the simulation, a **score is calculated**.

The attacker attempts to **maximize the score**.

The defender attempts to **minimize the score**.

You will implement:

- Minimax Search
- Alpha-Beta Pruning

to determine the optimal outcome assuming both players act rationally.

## Network Resource Constraint

The network has a limited amount of **bandwidth**.

Every action consumes bandwidth.

Rules:

- Bandwidth decreases when an action is performed.
- An action cannot be performed if its bandwidth cost is greater than the remaining bandwidth.
- Bandwidth cannot go below zero.

This constraint affects which actions are valid during the search.

## Input File Overview

This lab uses four input files.

---

### 1. network_state.txt

Contains the initial state of the network.

Format:

access_level  
system_security  
alert_level  
bandwidth  
rounds

Example:

3  
8  
1  
10  
5

Meaning:

access_level = 3  
system_security = 8  
alert_level = 1  
bandwidth = 10  
game will run for 5 rounds

---

### 2. attacker_actions.txt

Actions available to the attacker.

Format:

number_of_actions

action_name access_delta security_delta alert_delta bandwidth_cost

Example:

6  
scan_network 1 -1 1 1  
exploit_vulnerability 3 -3 2 3  
lateral_movement 2 -1 1 2  
credential_theft 2 -2 1 2  
privilege_escalation 3 -2 2 3  
install_backdoor 2 -1 3 2

---

### 3. defender_actions.txt

Actions available to the defender.

Format:

number_of_actions

action_name access_delta security_delta alert_delta bandwidth_cost

Example:

6  
patch_system -2 3 0 1  
increase_monitoring 0 2 2 2  
block_access -3 1 1 2  
reset_credentials -2 2 1 2  
network_isolation -3 2 2 3  
deploy_firewall -1 3 1 1

---

### 4. weights.txt

Defines how the final score is calculated.

Format:

access_weight  
security_weight  
alert_weight

Example:

2.5  
-1.8  
-1.2

Score formula:

score =  
(access_level × access_weight)  
+ (system_security × security_weight)  
+ (alert_level × alert_weight)

## Learning Objectives

By completing this lab, students will:

- Implement the Minimax algorithm
- Implement Alpha-Beta pruning
- Model a decision problem as a game tree
- Represent game states using classes
- Read structured data from files

## Step 1

Review the classes used to represent actions and network states.

In [275]:
class Action:

    def __init__(self, name, access_delta, security_delta, alert_delta, bandwidth_cost):

        self.name = name
        self.access_delta = access_delta
        self.security_delta = security_delta
        self.alert_delta = alert_delta
        self.bandwidth_cost = bandwidth_cost


class NetworkState:

    def __init__(self, access, security, alert, bandwidth, round_number, attacker_turn):

        self.access = access
        self.security = security
        self.alert = alert
        self.bandwidth = bandwidth
        self.round_number = round_number
        self.attacker_turn = attacker_turn


    def copy(self):
        """
        Create and return a deep copy of the state.
        """
        c = NetworkState(access = self.access,
        security = self.security,
        alert = self.alert,
        bandwidth = self.bandwidth,
        round_number = self.round_number,
        attacker_turn = self.attacker_turn)

        return c

## Step 2

Implement functions to read the input files.

In [276]:
def read_network_state(filename):

    """
    Read the initial network state from file.
    """

    with open(filename, 'r') as f:
        content = tuple(map(int, f.read().split()))

        return content
         


def read_actions(filename):

    """
    Read all actions from file and return a list of Action objects.
    """

    actions = []

    with open(filename, 'r') as f:
        for line in f:
            if not line.strip() or len(line.strip()) == 1:
                continue

            
            name, access_delta, security_delta, alert_delta, bandwidth_cost = line.strip().split()

            action = Action(name, int(access_delta), int(security_delta), int(alert_delta), int(bandwidth_cost))
            actions.append(action)

    return actions


def read_weights(filename):

    """
    Read evaluation weights from file.
    """

    with open(filename, 'r') as f:
        content = list(map(float, f.read().split()))

        return content

## Step 3

Implement the state transition function.

This function applies an action and returns the new network state.

In [277]:
def apply_action(state, action):

    """
    PSEUDOCODE

    - create a copy of the current state
    - modify the state according to the action:
        * adjust access, security, and alert levels
        * reduce available bandwidth
    - ensure no value is negative
    - switch the player turn
    - increment the round number
    - return the new state
    """

    s = state.copy()

    s.access = s.access + action.access_delta
    s.security = s.security + action.security_delta
    s.alert = s.alert + action.alert_delta
    s.bandwidth = s.bandwidth - action.bandwidth_cost
    s.round_number = s.round_number + 1
    s.attacker_turn = not s.attacker_turn

    if s.access < 0 or s.security < 0 or s.alert < 0 or s.bandwidth < 0:
        return None

    return s

## Step 4

Implement the evaluation function.

This function calculates the score when the game reaches the final round.

In [278]:
def evaluate_state(state, weights):

    """
    Compute and return the score of the given state.
    """

    score =  (state.access * weights[0]) + (state.security * weights[1]) + (state.alert * weights[2])

    return score

## Step 5

Implement the **Minimax algorithm**.

Attacker = MAX player  
Defender = MIN player

The algorithm should recursively explore the game tree until a terminal state is reached.

In [279]:
def minimax(state, attacker_actions, defender_actions, weights, max_rounds):

    """
    PSEUDOCODE

    - if the state is terminal (last round or no valid moves):
        return the evaluated score

    - if it is the attacker’s turn (MAX):
        initialize best score to negative infinity
        for each valid attacker action:
            apply the action to get a new state
            recursively call minimax on the new state
            update best score if higher
        return best score

    - if it is the defender’s turn (MIN):
        initialize best score to positive infinity
        for each valid defender action:
            apply the action to get a new state
            recursively call minimax on the new state
            update best score if lower
        return best score
    """

    if state.round_number == max_rounds:
        return evaluate_state(state, weights)

    if state.attacker_turn:
        b_score = float("-inf")
        for action in attacker_actions:
            n_state = apply_action(state, action)
            n_score = 0
            if n_state is not None:
                n_score = minimax(n_state, attacker_actions, defender_actions, weights, max_rounds)
            else:
                n_score = evaluate_state(state, weights)

            b_score = max(b_score, n_score)

        return b_score

    else:
        b_score = float("inf")
        for action in defender_actions:
            n_state = apply_action(state, action)
            n_score = 0
            if n_state is not None:
                n_score = minimax(n_state, attacker_actions, defender_actions, weights, max_rounds)
            else:
                n_score = evaluate_state(state, weights)

            b_score = min(b_score, n_score)

        return b_score

## Step 6

Implement Alpha-Beta pruning.

Alpha-Beta pruning improves Minimax by skipping branches that cannot affect the final result.

alpha = best value for MAX  
beta = best value for MIN

In [280]:
nodes_explored = 0

def update_nodes_explored():
    global nodes_explored
    nodes_explored += 1


def alphabeta(state, attacker_actions, defender_actions, weights, max_rounds, alpha, beta):

    """
    PSEUDOCODE

    - increment the node counter
    - if the state is terminal:
        return the evaluated score

    - if it is the attacker’s turn (MAX):
        initialize best score to negative infinity
        for each valid attacker action:
            apply the action to get a new state
            recursively call alphabeta on the new state
            update best score
            update alpha if best score > alpha
            if alpha >= beta:
                prune remaining actions
                break
        return best score

    - if it is the defender’s turn (MIN):
        initialize best score to positive infinity
        for each valid defender action:
            apply the action to get a new state
            recursively call alphabeta on the new state
            update best score
            update beta if best score < beta
            if alpha >= beta:
                prune remaining actions
                break
        return best score
    """

    update_nodes_explored()

    if state.round_number == max_rounds:
        return evaluate_state(state, weights)

    if state.attacker_turn:
        b_score = float("-inf")
        for action in attacker_actions:
            n_state = apply_action(state, action)
            n_score = 0
            if n_state is not None:
                n_score = alphabeta(n_state, attacker_actions, defender_actions, weights, max_rounds, alpha, beta)
            else:
                n_score = evaluate_state(state, weights)

            b_score = max(b_score, n_score)

            if b_score > alpha:
                alpha = b_score
            
            if alpha >= beta:
                break

        return b_score

    else:
        b_score = float("inf")
        for action in defender_actions:
            n_state = apply_action(state, action)
            n_score = 0
            if n_state is not None:
                n_score = alphabeta(n_state, attacker_actions, defender_actions, weights, max_rounds, alpha, beta)
            else:
                n_score = evaluate_state(state, weights)

            b_score = min(b_score, n_score)

            if b_score < beta:
                beta = b_score
            
            if alpha >= beta:
                break

        return b_score

## Step 7

Run the output writer.

Do NOT modify this code.

In [281]:
def write_output(score, nodes, filename="output.txt"):

    result = "Network Security Simulation Results\n"
    result += "----------------------------------\n"
    result += f"Final Score: {score}\n"
    result += f"Nodes Explored: {nodes}\n"

    print(result)

    with open(filename, "w") as file:
        file.write(result)

## Step 8

Run the main method.

Do NOT modify this code.

In [282]:
def main():

    access, security, alert, bandwidth, rounds = read_network_state("network_state.txt")

    attacker_actions = read_actions("attacker_actions.txt")
    defender_actions = read_actions("defender_actions.txt")

    weights = read_weights("weights.txt")

    state = NetworkState(access, security, alert, bandwidth, 0, True)

    score = alphabeta(
        state,
        attacker_actions,
        defender_actions,
        weights,
        rounds,
        float("-inf"),
        float("inf")
    )

    write_output(score, nodes_explored)


if __name__ == "__main__":
    main()

Network Security Simulation Results
----------------------------------
Final Score: -11.0
Nodes Explored: 1346



## Step 9

Answer the following questions.

1. Why does the Minimax algorithm explore many nodes?

2. How does Alpha-Beta pruning reduce the number of nodes explored?

3. Why does Alpha-Beta pruning not change the final decision?

4. What happens if the number of rounds increases significantly?

In [283]:
# Answers
# The minimax algo explores more nodes then in alphabeta because in alphabeta, if the next actions aren't gonna effect the value, meaning optimal value is found, it stops but there is no such check in minimax and it searches more nodes
# Because in alphabeta, if the next actions aren't gonna effect the value, meaning optimal value is found, it stops and not searches next action paths
# Because both give optimized value. alphabeta just stops exploring if action paths won't give more optimal value
# Then paths will grow exponentially and the search space will keep increasing, increasing computation used.